In [19]:
import pandas as pd
import glob

# Get a list of all CSV files in the directory
csv_files = glob.glob('training_data/flipkart//*.csv')

# Create a list to hold the dataframes
dfs = []

# Read each CSV file and append it to the list of dataframes
for csv_file in csv_files:
	df = pd.read_csv(csv_file)
	dfs.append(df)

	dfs.append(df)

# Concatenate all dataframes in the list
combined_df = pd.concat(dfs, ignore_index=True)

# Print the combined dataframe
# Extract a specific column
specific_column = combined_df['name'].dropna()

# Drop rows with non-existing values in the specific column
# todo wirte to a csv file
shuffled_df = specific_column.sample(frac=1).reset_index(drop=True)

# todo write the first 15000 rows of shuffled df to a txt file with a string on each line
# how t get each row as a string
with open('flipkart.txt', 'w') as f:
	for i in range(0, 15000):
		f.write(f"{shuffled_df[i]}\tTrue\n")




0        Trendy Lexi 6 Tier (Brown) Multi-Purpose Stora...
1        GLORIEUX ART Triangle Shape End, Coffee Table ...
2        Flipkart Perfect Homes Studio KR-02-White Engi...
3        ROYAL METAL FURNITURE Mattress Not Included Te...
4        Wooden Art & Toys HUT SHAPED WALL SHELVES Engi...
                               ...                        
28795                Modway Solid Wood 6 Seater Dining Set
28796          Allie Wood Sheesham Solid Wood Dining Chair
28797        Nilkamal Riva Engineered Wood 2 Door Wardrobe
28798    ELTOP Lifestyle Rolex L-Shape Leatherette 6 Se...
28799    Flipkart Perfect Homes Studio Wall Mounted TV ...
Name: name, Length: 28800, dtype: object

In [29]:
# Open the file
with open('training_data/filtered.txt', 'r') as file:
    lines = file.readlines()

# Remove leading and trailing whitespace from each line
lines = [line.strip() for line in lines]

# Write the lines back to the file
with open('training_data/filtered.txt', 'w') as file:
    file.write('\n'.join(lines))

In [35]:
import pandas as pd
import glob
# concat all .txt files into one dataframe

# Get a list of all .txt files in the directory
txt_files = glob.glob('training_data/*.txt')


txt_files
# # Create a list to hold the dataframes
dfs = []

# Read each .txt file and append it to the list of dataframes
for txt_file in txt_files:
	df = pd.read_csv(txt_file, sep='\t', names=["text", "label"])
	dfs.append(df)

# # Concatenate all dataframes in the list
combined_df = pd.concat(dfs, ignore_index=True)

# # Print the combined dataframe
print(combined_df)
count_true = (combined_df['label'] == True).sum()
print(count_true)


                                                    text  label
0                                            RRP $754.42  False
1                                         Massage Tables  False
2                                          (Was $623.28)  False
3                                             Save $263!  False
4                                          (Was $219.37)  False
...                                                  ...    ...
56975  Display Wall Shelves Industrial DIY Pipe Shelf...   True
56976  610x1219mm Commercial Stainless Steel Kitchen ...   True
56977                        Monitor Stand Desktop Riser   True
56978                    Wooden Shoe Organiser - Natural   True
56979                         2 Tier Shoe Cabinet - Wood   True

[56980 rows x 2 columns]
26516


In [40]:
shuffled_df = combined_df.sample(frac=1).reset_index(drop=True)

In [41]:
shuffled_df

,text,label
0,First ever Gaming Chair,False
1,Spring Daze Friendship 5 Pack,False
2,"Bed frame w/storage+slatted bedbase, Twin",True
3,22 DECEMBER 2023,False
4,THE TECHNOLOGY,False
...,...,...
56975,MBTC Clark Metal Outdoor Chair,True
56976,classiconline Engineered Wood Display Unit,True
56977,Tennis,False
56978,"Utility cart, 22 1/2x15 3/8x33 7/8 """,True


In [42]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

2023-12-23 19:38:56.798997: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2023-12-23 19:38:56.997597: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2023-12-23 19:38:56.997642: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2023-12-23 19:38:57.037281: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2023-12-23 19:38:57.122032: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2023-12-23 19:38:57.123031: I tensorflow/core/platform/cpu_feature_guard.cc:1

In [43]:
def process_data(row):
    text = row['text']
    text = str(text)
    text = ' '.join(text.split())

    encodings = tokenizer(text,truncation=True)

    label = 0
    if row['label'] == 1:
        label += 1

    encodings['label'] = label
    encodings['text'] = text

    return encodings

In [44]:
print(process_data(shuffled_df.iloc[0]))

{'input_ids': [101, 2034, 2412, 10355, 3242, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1], 'label': 0, 'text': 'First ever Gaming Chair'}


In [46]:
processed_data = []
df = shuffled_df

for i in range(len(df)):
    processed_data.append(process_data(df.iloc[i]))

In [48]:
count = 0
for data in processed_data:
	if data['label'] == 0:
		count += 1
print(count)

30464


In [49]:
df.count()

text     56978
label    56974
dtype: int64

In [50]:
df["label"]

0        False
1        False
2         True
3        False
4        False
         ...  
56975     True
56976     True
56977    False
56978     True
56979    False
Name: label, Length: 56980, dtype: object

In [51]:
from sklearn.model_selection import train_test_split

new_df = pd.DataFrame(processed_data)

train_df, valid_df = train_test_split(
    new_df,
    test_size=0.19,
    random_state=2022
)

In [52]:
import pyarrow as pa
from datasets import Dataset

train_hg = Dataset(pa.Table.from_pandas(train_df)).shuffle(seed=2022)
valid_hg = Dataset(pa.Table.from_pandas(valid_df)).shuffle(seed=2022)

In [53]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly i

In [54]:

import torch
from transformers import TrainingArguments, Trainer
import torch
from transformers import TrainingArguments, Trainer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model.to(device)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.02,
    logging_dir='./logs',  # Specify GPU device
)

trainer = Trainer(
    model=model,
    args=training_args,
	train_dataset=train_hg,
    eval_dataset=valid_hg,
    tokenizer=tokenizer
)

In [55]:
trainer.train()

The following columns in the training set don't have a corresponding argument in `BertForSequenceClassification.forward` and have been ignored: text, __index_level_0__. If text, __index_level_0__ are not expected by `BertForSequenceClassification.forward`,  you can safely ignore this message.
/home/oprisorandrei/.local/lib/python3.10/site-packages/transformers/optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 46153
  Num Epochs = 3
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 8655
  Number of trainable parameters = 109483778


  0%|          | 0/8655 [00:00<?, ?it/s]

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [79]:
from torch.utils.data import DataLoader

model = AutoModelForSequenceClassification.from_pretrained('./model')
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

data_loader = DataLoader(valid_hg, batch_size=16, shuffle=True)



The following columns in the evaluation set don't have a corresponding argument in `BertForSequenceClassification.forward` and have been ignored: __index_level_0__, text. If __index_level_0__, text are not expected by `BertForSequenceClassification.forward`,  you can safely ignore this message.
***** Running Evaluation *****
  Num examples = 3750
  Batch size = 64
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


  0%|          | 0/59 [00:00<?, ?it/s]

{'eval_loss': 0.6953178644180298,
 'eval_runtime': 78.0917,
 'eval_samples_per_second': 48.02,
 'eval_steps_per_second': 0.756}

In [80]:
from sklearn.metrics import classification_report

model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0): BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, element

In [94]:
from transformers import AutoModelForSequenceClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

new_model = AutoModelForSequenceClassification.from_pretrained('./results/checkpoint-4000/').to(device)

loading configuration file ./results/checkpoint-4000/config.json
Model config BertConfig {
  "_name_or_path": "./results/checkpoint-4000/",
  "architectures": [
    "BertForSequenceClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "problem_type": "single_label_classification",
  "torch_dtype": "float32",
  "transformers_version": "4.26.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading weights file ./results/checkpoint-4000/pytorch_model.bin
All model checkpoint weights were used when initializing BertForSequenceClassification.

All the weights of B

In [95]:
new_model.save_pretrained('./model4epochs/')

Configuration saved in ./model4epochs/config.json
Model weights saved in ./model4epochs/pytorch_model.bin


In [69]:
# Load the BERT model weights
plm = torch.load('./results/checkpoint-5000/pytorch_model.bin')

# Accessing a specific tensor
word_embeddings_weight = plm['classifier.weight']
len(word_embeddings_weight[1])

768

In [16]:

from transformers import AutoTokenizer

new_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

In [84]:
import torch
import numpy as np


def get_prediction(tokenizer, model, text):
	device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
	encoding = tokenizer(text, return_tensors='pt', truncation=True, padding='max_length', max_length=16)
	encoding = {k: v.to(device) for k, v in encoding.items()}

	outputs = model(**encoding)

	logits = outputs.logits
	probs = torch.nn.functional.softmax(logits, dim=-1)
	sigmoid = torch.nn.Sigmoid()
	probs = sigmoid(logits.squeeze().detach())
	probs = probs.detach().numpy()
	if np.argmax(probs, axis=-1) == 1:
		return True
	else:
		return False
		

In [85]:
def evaluate_model(model, dataset, tokenizer):
	texts = dataset['text']
	true_labels = dataset['label']

	predictions = [get_prediction(tokenizer, model, text) for text in texts]

	report = classification_report(true_labels, predictions)
	return report
	

In [87]:
report = evaluate_model(new_model, valid_df, new_tokenizer)

In [88]:
print (report)

              precision    recall  f1-score   support

           0       1.00      0.99      1.00      2291
           1       0.99      1.00      0.99      1459

    accuracy                           1.00      3750
   macro avg       1.00      1.00      1.00      3750
weighted avg       1.00      1.00      1.00      3750



In [60]:
import time
start = time.time()
print(get_prediction(new_tokenizer, new_model, "Telephone 65x20 cm"))
print (time.time() - start)

[0.00911242 0.99412596]
0.04905390739440918


In [49]:
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup as bs
import torch
import numpy as np
import threading
from urllib.parse import urlparse
from transformers import AutoTokenizer, AutoModelForSequenceClassification

options = Options()
service = Service(ChromeDriverManager().install())



class crawler:
	def __init__(self):
		self.lock = None
		self.tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
		device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
		self.model = AutoModelForSequenceClassification.from_pretrained('./model').to(device)

	def get_prediction(self, tokenizer, model, text):
		device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
		encoding = tokenizer(text, return_tensors='pt', truncation=True)
		encoding = {k: v.to(device) for k, v in encoding.items()}

		outputs = model(**encoding)

		logits = outputs.logits
		probs = torch.nn.functional.softmax(logits, dim=-1)
		sigmoid = torch.nn.Sigmoid()
		probs = sigmoid(logits.squeeze().detach())
		probs = probs.detach().numpy()
		if np.argmax(probs, axis=-1) == 1:
			return text

	def scraper(self, urls):
		products = []
		driver = webdriver.Chrome(service=service, options=options)
		driver.set_page_load_timeout
		for url in urls:
			print(f"Crawling {url}...")
			products = []
			try:
				driver.get(url)
				# Get the page source
				products = []
				page_source = driver.page_source
				soup = bs(page_source, 'html.parser')
				links = soup.find_all('a')
				headers = soup.find_all(['h1','h2','h3','h4','h5','h6', 'p'])

				for header in headers:
					if self.get_prediction(self.tokenizer, self.model, header.get_text(strip=True)) is not None:
						products.append(header.get_text(strip=True))

				for link in links:
					text = link.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'p'])
					if text is None:
						texts = link.get_text(separator=' | ',strip=True).split(' | ')
						for text in texts:
							if self.get_prediction(self.tokenizer, self.model, text) is not None:
								products.append(text)
								break
			except Exception as e:
				print(f"Exception occurred while crawling: {str(e)}")
				continue
			
			if products and self.lock is not None:
				with self.lock:
					with open('products.txt', 'a') as f:
						f.write(urlparse(url).netloc + '\n')
						for product in products:
							f.write(product + '\n')
			elif products:
				with open('products.txt', 'a') as f:
					f.write(urlparse(url).netloc + '\n')
					for product in products:
						f.write(product + '\n')
		driver.quit()



	def crawl(self, urls, threaded, num_workers = None):
		if threaded:
			threads = []
			self.lock = threading.Lock()
			for _ in range(num_workers):
				t = threading.Thread(target=self.scraper, args=(urls,))
				t.start()
				threads.append(t)
			
			for thread in threads:
				thread.join()
		else:
			self.scraper(urls)
	 